<a href="https://colab.research.google.com/github/sromerar/omics-tutorials/blob/main/atlas/notebooks/02_atlas_palantir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ATLAS: Trajectory Inference with Palantir

Annotated reproduction of the official ATLAS Tutorial 2, using Palantir to infer developmental trajectories from paired single-cell RNA expression and chromatin-derived gene activity.

This tutorial uses a multimodal SHARE-seq dataset of mouse hair-follicle development, including hair follicle stem cells, transit-amplifying cells, and differentiated progeny.

## 1. Environment setup

This notebook is run in Google Colab using the 2026.07 past runtime (Python 3.12.13). The current default Colab Python 3.13 runtime caused ATLAS dependencies to be built from source and was not suitable for this tutorial.

I install ATLAS and the core Python packages needed for the analysis. NumPy and pandas are pinned to the versions that previously worked reliably with ATLAS in this environment.

In [1]:
import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
import sys

!{sys.executable} -m pip install \
    "numpy==2.0.2" \
    "pandas==2.2.3" \
    "atlas-smilies==1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of anndata to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of scanpy to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.0/245.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.2 MB/s eta 

### Dependency compatibility note

The installation completes successfully, but Google Colab reports dependency-version warnings. In particular, this runtime expects `pandas==2.2.2`, whereas this notebook uses `pandas==2.2.3`, and some Colab packages expect a newer version of `jinja2` than the one installed through the ATLAS dependency stack.

These are dependency warnings rather than installation errors. I therefore verify that ATLAS and its core dependencies import successfully before proceeding with the analysis.

### Verify the installation

I verify that ATLAS and its main dependencies import successfully in the current Colab environment before proceeding with the analysis.

In [3]:
import atlas
import numpy as np
import pandas as pd
import scanpy as sc
import muon as mu

print("ATLAS imported successfully")
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Scanpy:", sc.__version__)
print("muon:", mu.__version__)

ATLAS imported successfully
NumPy: 2.0.2
pandas: 2.2.3
Scanpy: 1.12
muon: 0.1.9


## 2. Load and prepare the SHARE-seq dataset

The official ATLAS Palantir tutorial uses a preprocessed multimodal SHARE-seq dataset of mouse hair-follicle cells. I first define a fixed random seed for reproducibility and create the local directory where the tutorial data will be stored.

In [4]:
import os

seed = 42
rng = np.random.default_rng(seed)
np.random.seed(seed)

data_path = os.path.join(os.getcwd(), "data")
os.makedirs(data_path, exist_ok=True)

print("Data directory:", data_path)
print("Random seed:", seed)

Data directory: /content/data
Random seed: 42


### SHARE-seq mouse hair-follicle dataset

This tutorial uses a preprocessed SHARE-seq dataset of mouse hair-follicle development. The dataset contains paired gene-expression and chromatin-derived gene-activity measurements and includes hair follicle stem/progenitor populations, transit-amplifying cells, and differentiated lineages such as medulla, cuticle/cortex, and inner root sheath cells.

The official ATLAS Tutorial 2 begins from a preprocessed `hair.h5mu` multimodal object rather than repeating the raw-data preprocessing performed in Tutorial 1.

### Data availability and preprocessing

The official ATLAS Tutorial 2 starts from a preprocessed `hair.h5mu` object. This file is not distributed directly with the public ATLAS repository, so I reconstruct it from the original SHARE-seq mouse hair-follicle data deposited in GEO, following the preprocessing workflow provided in the authors' `atlas-experiments` repository.

Because this preprocessing requires the original RNA, ATAC, cell-annotation, peak, barcode, and fragment files, it is performed here before reproducing the Palantir trajectory-inference analysis.

In [5]:
data.write(os.path.join(output_path, "hair.h5mu"))

NameError: name 'data' is not defined